# 13 - When does accDM become indistinguishable from CDM?

The accelerated-DM daughter is born with a velocity kick `v = sqrt(eta(eta+2))/(1+eta)`
set by the energy boost `eta_acc`. Adopting the physical relation **`eta = 1e11 / m`**
(`m` = daughter mass in GeV), a heavier daughter has smaller `eta`, free-streams less,
and is colder. This notebook sweeps `m` upward and finds the mass above which accDM's
matter power `P(k)` and lensed CMB spectra match "CDM" within our thresholds.

Two references ("CDM"): (1) the **cold limit** of the same model (`eta -> 0`), isolating
free-streaming; (2) **plain LCDM** (no acc species). Two metric families: (A) a fixed
fractional tolerance on the spectra, and (B) a cosmic-variance-limited chi^2 detectability.

Daughter on exact quadrature (fluid closure is unusable). Hybrid notebook: inline unit
asserts for the pure metrics (nbmake) + a CLASS scan + diagnostic plots. Run in the
`accDM` classy environment: `pytest --nbmake notebooks_test/13_test_accDM_CDM_indistinguishability.ipynb`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from classy import Class

plt.rcParams.update({
    'mathtext.fontset': 'stix', 'font.family': 'serif', 'font.size': 11,
    'axes.labelsize': 12, 'legend.fontsize': 9, 'lines.linewidth': 1.5, 'figure.dpi': 300})
qual_colors = ['#377eb8', '#ff7f00', '#4daf4a', '#f781bf', '#984ea3']

# ---- Planck 2018 base cosmology ----------------------------------
omega_b, omega_cdm0 = 0.022383, 0.12011
A_s, n_s, tau_reio, H0 = 2.1005829616811546e-9, 0.96605, 0.0543, 67.32
base_params = {'omega_b': omega_b, 'omega_cdm': omega_cdm0, 'H0': H0,
               'A_s': A_s, 'n_s': n_s, 'tau_reio': tau_reio}

# ---- fixed accDM decay sector ------------------------------------
KAPPA, A_T = 2.0, 0.13
A_REC = 1.0 / (1.0 + 1090.0)
ETA_COLD = 1e-12                     # "cold limit" eta

# ---- scan axes ---------------------------------------------------
F_SEQ = [0.01, 0.05, 0.1]
MASS_GRID = np.logspace(11, 19, 12)  # GeV  -> eta ~ 1 down to ~1e-8

# ---- observables -------------------------------------------------
K_NODES = np.logspace(-3, 0.0, 60)   # 1/Mpc, sub-horizon at z=0
L_MAX = 2500
Q_BINS = 250                          # daughter momentum bins (accuracy vs speed knob)

PREC_COMMON = {'evolver': 0, 'reionization_z_start_max': 80, 'background_Nloga': 2001}
PREC_PK  = {**PREC_COMMON, 'output': 'mPk', 'P_k_max_1/Mpc': 1.0, 'z_max_pk': 0.0}
PREC_CMB = {**PREC_COMMON, 'output': 'tCl,pCl,lCl,mPk', 'lensing': 'yes',
            'l_max_scalars': L_MAX, 'P_k_max_1/Mpc': 1.0, 'z_max_pk': 0.0}

# ---- survey / detectability assumptions --------------------------
V_SURVEY = 100.0**3                   # (100 Mpc)^3 fiducial; edit to taste
F_SKY = 0.7

def eta_of(m):
    return 1e11 / m


In [ ]:
def ocdm_rescaled(f_acc):
    """CDM density rescaled for the decayed daughter, as in notebook 5."""
    return omega_cdm0 * (1 + f_acc*(1 - A_REC**KAPPA)/(1 + (A_REC/A_T)**KAPPA))**(-1)

def lcdm_params():
    """Plain LCDM, no acc species; a massive-nu sector matched to the accDM runs."""
    p = dict(base_params); p.update(PREC_CMB)
    p.update({'N_ncdm': 1, 'deg_ncdm': 3, 'm_ncdm': 0.02, 'T_ncdm': 0.71611,
              'ncdm_quadrature_strategy': 0, 'ncdm_N_momentum_bins': 15, 'N_ur': 0.00441})
    return p

def _accdm_common(f_acc, m, eta):
    p = dict(base_params); p.update(PREC_CMB)
    p.update({'omega_cdm': ocdm_rescaled(f_acc),
              'vary_Gamma_acc': 'yes', 'kappa_acc': KAPPA, 'a_t_acc': A_T,
              'f_acc': f_acc, 'eta_acc': eta,
              'm_acc_in_GeV': m, 'm_cdm_in_GeV': m,
              'N_ncdm': 2, 'deg_ncdm': '3, 1',
              'm_ncdm': '0.02, {:.6e}'.format(m*1e9),
              'T_ncdm': '0.71611, 1', 'ncdm_quadrature_strategy': '0, 4',
              'ncdm_N_momentum_bins': '15, {:d}'.format(Q_BINS), 'N_ur': 0.00441,
              'ncdm_fluid_approximation': 0,
              'ncdm_fluid_trigger_rho_accDM_over_rho_dcdm': 10})
    return p

def accdm_params(f_acc, m):
    return _accdm_common(f_acc, m, eta_of(m))

def coldlimit_params(f_acc):
    return _accdm_common(f_acc, MASS_GRID[-1], ETA_COLD)


In [ ]:
# structural sanity: mapping + required keys present, exact quadrature enforced
assert abs(eta_of(1e11) - 1.0) < 1e-12
assert abs(eta_of(1e18) - 1e-7) < 1e-15
_p = accdm_params(0.1, 1e15)
assert _p['ncdm_quadrature_strategy'].endswith('4')      # daughter exact
assert _p['ncdm_fluid_approximation'] == 0               # never fluid
assert _p['m_ncdm'].split(',')[1].strip() == '{:.6e}'.format(1e15*1e9)
assert _p['eta_acc'] == eta_of(1e15)
assert coldlimit_params(0.1)['eta_acc'] == ETA_COLD
print('Task 1 param builders OK')


In [ ]:
_cache = {}
def run(params, key):
    """Compute a CLASS model once; return P(k) on K_NODES and lensed Cl (l=2..L_MAX)."""
    if key in _cache:
        return _cache[key]
    M = Class(); M.set(params); M.compute()
    pk = np.array([M.pk(float(k), 0.0) for k in K_NODES])
    cl = M.lensed_cl(L_MAX)
    ell = cl['ell']
    sel = ell >= 2
    out = {'pk': pk, 'ell': ell[sel],
           'tt': cl['tt'][sel], 'te': cl['te'][sel], 'ee': cl['ee'][sel]}
    M.struct_cleanup(); M.empty()
    _cache[key] = out
    return out

def get_lcdm():
    return run(lcdm_params(), ('lcdm',))

def get_cold(f_acc):
    return run(coldlimit_params(f_acc), ('cold', f_acc))

def get_warm(f_acc, m):
    return run(accdm_params(f_acc, m), ('warm', f_acc, m))


## Smoke test: one warm run + LCDM reference

In [ ]:
_warm = get_warm(0.1, 1e13)          # eta = 0.01
_lcdm = get_lcdm()
assert _warm['pk'].shape == K_NODES.shape
assert _warm['tt'].shape == _warm['ell'].shape
assert np.all(np.isfinite(_warm['pk'])) and np.all(np.isfinite(_warm['tt']))
assert _warm['ell'][0] == 2 and _warm['ell'][-1] == L_MAX
print('Task 2 run/cache OK: pk[0]={:.3e}, tt[100]={:.3e}'.format(_warm['pk'][0], _warm['tt'][100]))


## Metrics (pure functions) + unit tests

In [ ]:
# --- unit tests (defined before the functions on purpose; run after next cell) ---
def _test_metrics():
    k = np.logspace(-3, 0, 60)
    a = np.ones_like(k) * 2.0
    # identical spectra -> zero on every metric
    assert pk_maxdev(a, a) == 0.0
    assert pk_significance(a, a, k, 1e6) == 0.0
    assert cl_maxdev(a, a) == 0.0
    # max fractional deviation is exact
    b = a.copy(); b[10] = a[10] * 1.05
    assert abs(pk_maxdev(b, a) - 0.05) < 1e-12
    assert abs(cl_maxdev(b, a) - 0.05) < 1e-12
    # pk_significance matches the hand formula on a flat 1% offset
    ref = np.ones_like(k); acc = ref * 1.01
    dk = np.gradient(k); Nm = k**2 * dk * 1e6 / (2*np.pi**2)
    want = np.sqrt(np.sum((0.01 / np.sqrt(2.0/Nm))**2))
    assert abs(pk_significance(acc, ref, k, 1e6) - want) < 1e-9
    # cmb_significance is zero for identical run-dicts
    d = {'ell': np.arange(2, 50), 'tt': np.ones(48), 'te': np.zeros(48), 'ee': np.ones(48)}
    assert cmb_significance(d, d) == 0.0
    print('metric unit tests PASSED')

In [ ]:
def pk_maxdev(pk_acc, pk_ref):
    return float(np.max(np.abs(pk_acc / pk_ref - 1.0)))

def pk_significance(pk_acc, pk_ref, k=K_NODES, V=V_SURVEY):
    dk = np.gradient(k)
    N_modes = k**2 * dk * V / (2.0 * np.pi**2)
    sigmaP_over_P = np.sqrt(2.0 / N_modes)
    resid = (pk_acc - pk_ref) / pk_ref / sigmaP_over_P
    return float(np.sqrt(np.sum(resid**2)))

def cl_maxdev(cl_acc, cl_ref):
    return float(np.max(np.abs(cl_acc / cl_ref - 1.0)))

def cmb_significance(acc, ref, f_sky=F_SKY):
    """Cosmic-variance-limited Knox chi over TT+EE+TE (cross-covariance neglected)."""
    ell = ref['ell']; norm = (2.0 * ell + 1.0) * f_sky
    tt_r, ee_r, te_r = ref['tt'], ref['ee'], ref['te']
    var_tt = 2.0 * tt_r**2 / norm
    var_ee = 2.0 * ee_r**2 / norm
    var_te = (te_r**2 + tt_r * ee_r) / norm
    sig2 = np.sum((acc['tt'] - tt_r)**2 / var_tt)
    sig2 += np.sum((acc['ee'] - ee_r)**2 / var_ee)
    sig2 += np.sum((acc['te'] - te_r)**2 / var_te)
    return float(np.sqrt(sig2))

In [ ]:
_test_metrics()

In [ ]:
def _test_threshold():
    m = np.logspace(11, 19, 9)
    vals = np.array([1.0, 0.8, 0.6, 0.4, 0.2, 0.1, 0.05, 0.02, 0.01])   # decreasing
    # cut between vals[4]=0.2 and vals[5]=0.1 -> mass between m[4] and m[5]
    t = threshold_mass(m, vals, 0.15)
    assert m[4] < t < m[5]
    # already below everywhere -> smallest mass
    assert threshold_mass(m, vals*1e-3, 0.15) == m[0]
    # never below -> inf
    assert threshold_mass(m, vals*1e3, 0.15) == np.inf
    print('threshold unit tests PASSED')


In [ ]:
def threshold_mass(masses, metric_values, cut):
    masses = np.asarray(masses, float); vals = np.asarray(metric_values, float)
    logm = np.log10(masses)
    below = vals < cut
    if below.all():
        return float(masses[0])
    if not below.any():
        return np.inf
    i = int(np.argmax(below))            # first index below the cut
    if i == 0:
        return float(masses[0])
    x0, x1, y0, y1 = logm[i-1], logm[i], vals[i-1], vals[i]
    xc = x0 + (cut - y0) * (x1 - x0) / (y1 - y0)
    return float(10**xc)


In [ ]:
_test_threshold()